## Get Physical Properties from AlphaFold Output

May need `openmmtools` and `openff-toolkit` from `conda-forge` and `omnia`

In [ ]:
# Install packages
%pip install -q numpy pandas matplotlib seaborn pathos biopython tdqm mdtraj rdkit MDAnalysis prolif
%pip install -q openmm openmmtools openff-toolkit

: 

## Setup

In [1]:
import prolif as plf
import MDAnalysis as mda
from MDAnalysis.analysis import contacts
from rdkit import Chem
from rdkit.Chem import AllChem
import os
import numpy as np

C:\Users\ryangustafson\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\MDAnalysis\topology\tables.py:52: DeprecationWarning: Deprecated in version 2.8.0
MDAnalysis.topology.tables has been moved to MDAnalysis.guesser.tables. This import point will be removed in MDAnalysis version 3.0.0
  warnings.warn(wmsg, category=DeprecationWarning)


In [ ]:
import os
from glob import glob
import numpy as np
import mdtraj as md
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

from Bio.PDB import PDBParser, PDBIO, Select

from openmm import app, unit, Platform
import openmm as mm
from openmmtools import mbar, states, alchemy
from openff.toolkit.topology import Molecule

import prolif as plf

from pathos.multiprocessing import ProcessingPool as Pool


ModuleNotFoundError: No module named 'openff'

In [9]:
class ChainSelect(Select):
    def __init__(self, chain_id):
        self.chain_id = chain_id
    def accept_chain(self, chain):
        return chain.id == self.chain_id

def split_chains(pdb_file):
    parser = PDBParser(QUIET=True)
    struct = parser.get_structure("complex", pdb_file)

    ligand_file = pdb_file.replace(".pdb", "_ligand.pdb")
    protein_file = pdb_file.replace(".pdb", "_protein.pdb")

    io = PDBIO()
    
    # Save ligand (chain A)
    io.set_structure(struct)
    io.save(ligand_file, select=ChainSelect("A"))
    
    # Save protein (chain B)
    io.save(protein_file, select=ChainSelect("B"))

    return ligand_file, protein_file


NameError: name 'Select' is not defined

In [ ]:
def build_system(ligand_pdb, protein_pdb):
    # Load protein
    protein = app.PDBFile(protein_pdb)
    modeller = app.Modeller(protein.topology, protein.positions)

    # Parametrize ligand
    ligand = Molecule.from_file(ligand_pdb)
    ligand_ff = ligand.to_topology().to_openmm()
    ligand_positions = ligand.conformers[0]

    modeller.add(ligand_ff, ligand_positions)

    # Add solvent
    modeller.addHydrogens()
    modeller.addSolvent(app.ForceField('amber14-all.xml', 'amber14/tip3p.xml'),
                        model='tip3p',
                        padding=1.0*unit.nanometers)

    forcefield = app.ForceField('amber14-all.xml', 'amber14/tip3p.xml')
    system = forcefield.createSystem(modeller.topology,
                                     nonbondedMethod=app.PME,
                                     nonbondedCutoff=1.0*unit.nanometers,
                                     constraints=app.HBonds)
    return modeller, system


In [ ]:
folder = "path_to_pdb_folder"
results = []

for pdb_file in tqdm(glob(folder + "/*.pdb")):
    ligand_pdb, protein_pdb = split_chains(pdb_file)

    modeller, system = build_system(ligand_pdb, protein_pdb)
    outprefix = pdb_file.replace(".pdb", "_md")

    traj = run_md(modeller, system, outprefix)

    analysis = analyze(protein_pdb, ligand_pdb, traj)
    analysis["complex"] = os.path.basename(pdb_file)

    results.append(analysis)

df = pd.DataFrame(results)
df.to_csv("md_results.csv", index=False)
df


In [ ]:
def analyze(protein_pdb, ligand_pdb, traj):
    traj_md = md.load(traj, top=protein_pdb)
    ligand_traj = md.load(traj, top=ligand_pdb)

    rmsd = md.rmsd(ligand_traj, ligand_traj, 0)
    hbonds = md.baker_hubbard(traj_md)

    return {
        "rmsd_mean": rmsd.mean(),
        "rmsd_max": rmsd.max(),
        "hbonds_count": len(hbonds)
    }


## MDAnalysis

In [7]:
import MDAnalysis as mda
from MDAnalysis.lib.distances import distance_array
import numpy as np

# Load your PDB file
pdb_file = "my_complex.pdb" # <-- Your PDB file here

try:
    u = mda.Universe(pdb_file)
except Exception as e:
    print(f"Error loading {pdb_file}: {e}")
    exit()

# --- 1. Define Your Specific Parts ---
sel_A = "chainID A"
sel_B = "chainID B"

group_A = u.select_atoms(sel_A)
group_B = u.select_atoms(sel_B)

if group_A.n_atoms == 0 or group_B.n_atoms == 0:
    print("Error: One or both selections resulted in 0 atoms.")
    print(f"Check if '{sel_A}' and '{sel_B}' are correct for your PDB.")
    exit()

print(f"Group A ({sel_A}): {group_A.n_atoms} atoms")
print(f"Group B ({sel_B}): {group_B.n_atoms} atoms")

# --- 2. Run Analysis (New Method) ---
distance_cutoff = 4.5

# Calculate the distance matrix between all atoms in group A
# and all atoms in group B.
# This creates a (n_atoms_A, n_atoms_B) array.
dist_matrix = distance_array(group_A.positions, group_B.positions)

# Find the indices (i, j) where the distance is less than the cutoff
# i = index in group_A, j = index in group_B
close_atom_indices = np.where(dist_matrix <= distance_cutoff)

# 'close_atom_indices' is a tuple of two arrays:
# (array_of_i_indices, array_of_j_indices)

# --- 3. Map Atom Indices to Residues ---

# Get the indices of the atoms in group_A that are close
close_atoms_A_indices = close_atom_indices[0]
# Get the indices of the atoms in group_B that are close
close_atoms_B_indices = close_atom_indices[1]

# Now, map these atom indices to their parent residues
# We use a set to store the residues so we only get unique ones
interacting_residues_A = set()
for atom_index in close_atoms_A_indices:
    interacting_residues_A.add(group_A[atom_index].residue)

interacting_residues_B = set()
for atom_index in close_atoms_B_indices:
    interacting_residues_B.add(group_B[atom_index].residue)

# --- 4. Print Results ---
print("\n--- MDAnalysis Contact Analysis (within 4.5 Å) ---")

print(f"\nResidues in Chain A interacting with Chain B:")
if interacting_residues_A:
    # Sort the residues by resid for a clean output
    sorted_residues = sorted(list(interacting_residues_A), key=lambda r: r.resid)
    print([res.resname + str(res.resid) for res in sorted_residues])
else:
    print("None found.")

print(f"\nResidues in Chain B interacting with Chain A:")
if interacting_residues_B:
    sorted_residues = sorted(list(interacting_residues_B), key=lambda r: r.resid)
    print([res.resname + str(res.resid) for res in sorted_residues])
else:
    print("None found.")

# --- Optional: Hydrogen Bond Analysis ---
# (This still has the same limitation: it needs hydrogens in the PDB)
try:
    from MDAnalysis.analysis.hydrogenbonds import HydrogenBondAnalysis
    
    # Select all protein atoms for H-bond analysis
    # We select 'protein' and 'protein' to find intra-protein H-bonds
    h = HydrogenBondAnalysis(u, sel1="protein", sel2="protein")
    h.run()
    
    # Filter for H-bonds specifically between Chain A and Chain B
    inter_chain_hbonds = []
    for bond in h.results.hbonds:
        donor_chain = u.atoms[bond[2]].chainID
        acceptor_chain = u.atoms[bond[0]].chainID
        
        if (donor_chain == 'A' and acceptor_chain == 'B') or \
           (donor_chain == 'B' and acceptor_chain == 'A'):
            inter_chain_hbonds.append(bond)

    print(f"\nFound {len(inter_chain_hbonds)} total inter-chain H-bonds (A-B).")
    # Note: This will likely be 0 if your PDB has no hydrogens.

except ImportError:
    print("\nHydrogen bond analysis requires an updated version of MDAnalysis.")
except Exception as e:
    print(f"\nCould not run H-bond analysis: {e}")

Group A (chainID A): 1520 atoms
Group B (chainID B): 1299 atoms

--- MDAnalysis Contact Analysis (within 4.5 Å) ---

Residues in Chain A interacting with Chain B:
['CYS1', 'THR2', 'CYS3', 'SER4', 'PRO5', 'PRO33', 'PHE34', 'GLY35', 'GLU62', 'SER64', 'GLU65', 'SER66', 'LEU67', 'CYS68', 'LYS71', 'ARG84', 'LEU94', 'CYS95', 'LYS123', 'LYS125', 'TYR128', 'TYR129', 'ASN148']

Residues in Chain B interacting with Chain A:
['PHE4', 'TYR73', 'ASP76', 'ASP79', 'GLY80', 'LEU81', 'LEU82', 'ALA83', 'HIS84', 'ALA85', 'PHE86', 'PRO87', 'ILE92', 'GLN93', 'GLY107', 'LYS108', 'GLN110', 'TYR112', 'VAL117', 'HIS120', 'GLU121', 'HIS124', 'ASP129', 'HIS130', 'PRO140', 'MET141', 'TYR142', 'ARG143', 'PHE144', 'GLU146']

Could not run H-bond analysis: HydrogenBondAnalysis.__init__() got an unexpected keyword argument 'sel1'. Did you mean 'self'?
